<a href="https://colab.research.google.com/github/MiriamSchirmer/nlp-llms-ws2627/blob/main/notebooks/2_Text_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6: Training and evaluating a text classifier

**Introduction to NLP and LLMs**, University of Konstanz, WS 2026/27

We will be working with **press releases by members of the US Congress**.

The data are 992 press releases by members of the US Congress (496 Democrats, 496 Republicans) from September 2022 to December 2023. They are a random sample from the press release data of the [Polarization Research Lab](https://polarizationresearchlab.org/).

We will use BoW and TF-IDF to **predict whether a press release was written by a Democrat or Republican**. This is a **binary classification** task: there are exactly two classes, `D` and `R`, and each press release gets one of them. Both classes have the same size, so guessing gives 50% accuracy.

## 1. Load the data

We load the file directly from the course repository on GitHub, so you do not need to download or upload anything.

In [ ]:
import pandas as pd   # tables (data frames)

csv_url = "https://raw.githubusercontent.com/MiriamSchirmer/nlp-llms-ws2627/main/data/press_releases_sample.csv"
df = pd.read_csv(csv_url)   # read the file from GitHub into a table

print(df.shape)              # (number of rows, number of columns)
print() # empty line for space
print(df["party"].value_counts())   # how many press releases per party?

In [ ]:
print(df.columns)   # the column names
df.head()           # the first five rows

Each row is one press release. We use the columns `party` (`D` for Democrat, `R` for Republican), `title` and `text`.

In [ ]:
example = df.iloc[0]          # the first press release (row 0)

print(example["party"])
print(example["title"])
print() # empty line for space
print(example["text"][:500])  # the first 500 characters of the text

## 2. Refresher: finding press releases with keywords

A quick look back at Session 3. The simplest way to find the press releases about a topic, here immigration, is a **keyword list** (a small dictionary) and a **regular expression**:

- `\b` is a word boundary: the match has to start at the beginning of a word.
- `\w*` allows any ending, so `immigr` matches *immigration*, *immigrant* and *immigrants*.
- `(?:a|b|c)` means *a or b or c*. `"|".join(keywords)` puts the words of the list together in this way.

In [ ]:
keywords = ["immigr", "migrant", "asylum", "border", "refugee", "deport"]

pattern = r"\b(?:" + "|".join(keywords) + r")\w*"
print(pattern)

df["immigration"] = df["text"].str.contains(pattern, case=False)   # True if the text contains at least one keyword

print(df["immigration"].sum(), "of", len(df), "press releases contain at least one keyword")
df.groupby("party")["immigration"].mean().round(2)   # share per party

Counts do not show how a word is used. A **keyword-in-context** (KWIC) list shows the words around each match.

In [ ]:
import re

def kwic(word, n=10):
    # show up to n matches of word, with 60 characters on each side
    shown = 0
    for text in df["text"]:
        for match in re.finditer(r"(?i)\b" + word + r"\w*", text):
            print("...", text[max(0, match.start() - 60):match.end() + 60].replace("\n", " "), "...")
            shown = shown + 1
            if shown == n:
                return

kwic("border")

Not every match has to be about immigration, and some texts about immigration use none of our words. The keyword list decides what you measure.

### ✏️ Your turn

1. Change the words in `keywords`. Run both cells again. How many press releases do you find now, and how does the share per party change?
2. Use `kwic(...)` with one of your new words. Are the matches about immigration?


**Note:** A keyword list only finds the words we thought of. A classifier learns which words matter from labelled examples. This is what we do next.

## 3. Train/test split

We train the classifier on the **training set** (80%) and evaluate it on a held-out **test set** (20%). **Only texts the classifier has not seen show how well it generalizes.**

- `test_size=0.2` puts 20% of the texts into the test set.
- `stratify=df["party"]` keeps the same share of D and R in both parts.
- `random_state=42` makes the random split the same every time we run the cell.

**Note**: we skip the **validation set**. It is used to choose settings (such as `min_df`) or to choose between models, so that the test set is used only once, at the end.

In [ ]:
# scikit-learn: classic machine learning
from sklearn.model_selection import train_test_split   # split the data into training and test set

# split the press releases: 80% training, 20% test
train, test = train_test_split(df, test_size=0.2, stratify=df["party"], random_state=42)

print(len(train), "training texts")
print(len(test), "test texts")

## 4. Text as features

Machine learning models do not work directly on text. We need numbers.

### Bag of words

A **bag of words** counts how often each word occurs in a document. Word order is ignored. All documents together give a **term-document matrix**: documents in rows, words (features) in columns.

We use the four headlines from the slides:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer   # texts -> word counts (bag of words)

headlines = ["the economy is collapsing", "the economy is growing", "the economy is stable", "the minister resigns"]

counter = CountVectorizer()
counts = counter.fit_transform(headlines)   # learn the vocabulary and count the words
words = counter.get_feature_names_out()     # the words (one column each)

pd.DataFrame(counts.toarray(), columns=words, index=headlines)   # the term-document matrix

### TF-IDF

- **Term frequency (TF)**: how often a word appears in a document.
- **Document frequency (DF)**: in how many documents a word appears.
- **TF-IDF** = TF × IDF, with IDF = log(N / DF). Words that occur in many documents get a lower weight.

With counts, all four words of headline 1 count the same. With TF-IDF, *collapsing* gets the highest weight and *the* the lowest.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer   # texts -> TF-IDF weights

tfidf = TfidfVectorizer()
weights_tfidf = tfidf.fit_transform(headlines)   # learn the vocabulary and compute TF-IDF

pd.DataFrame(weights_tfidf.toarray(), columns=words, index=headlines).round(2)

[🤓 More info on the weights: scikit-learn uses a slightly different formula than the slides, so the numbers are not the same, but the order is. It also scales each row to length 1, so a weight depends on the other words in the same headline: *stable* and *resigns* both occur in one headline only, but *resigns* (0.66) shares its headline with a second rare word, *minister*, while *stable* (0.69) is the only rare word in its headline.]

Now the same for our training texts. `stop_words="english"` removes stopwords (Session 3), and `min_df=2` (minimum document frequency) keeps only words that occur in at least two texts. Words that occur in one text only, such as typos or rare names, are dropped.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=2)
matrix = tfidf.fit_transform(train["text"])

print(matrix.shape)   # (number of texts, number of words in the vocabulary)

## 5. Logistic regression: a two-step pipeline

1. **Step 1: TF-IDF** turns the text into numbers. No labels are involved.
2. **Step 2: logistic regression** learns one weight per word from the labels. Labels enter only in this step, and only during training.

`make_pipeline` puts both steps together. `.fit()` trains the model: it starts with some weights, measures the gap between predictions and true labels (the **loss**), and changes the weights until the loss stops improving. `max_iter=1000` allows enough rounds for this.

Note: scikit-learn uses **regularization** by default. It penalizes big weights, so the model does not just memorize rare words of the training texts (overfitting).

In [ ]:
from sklearn.linear_model import LogisticRegression   # the classifier
from sklearn.pipeline import make_pipeline            # chains step 1 and step 2

model = make_pipeline(TfidfVectorizer(stop_words="english", min_df=2), LogisticRegression(max_iter=1000))
model.fit(train["text"], train["party"])   # learn the weights from the training set

For a new text, the model adds up the weights of its words (the **score**). The **sigmoid** function turns the score into a **probability** between 0 and 1. A **threshold** of 0.5 turns the probability into a class: above 0.5 → R, below → D.

`predict_proba` shows the probabilities, `predict` the classes. Here for the first five test texts:

In [ ]:
probabilities = model.predict_proba(test["text"])   # [probability for D, probability for R]
predictions = model.predict(test["text"])           # D or R (threshold 0.5)

print(probabilities[:5].round(2))
print(predictions[:5])

## 6. Evaluation

We compare the predictions with the true parties of the test set. We treat R as the "positive" class.

- **Accuracy**: share of correct predictions.
- **Precision** for R: of all texts predicted as R, what fraction are really R?
- **Recall** for R: of all real R texts, what fraction are predicted as R?
- **F1**: combines precision and recall, 2·P·R / (P + R).

Why not only accuracy? With imbalanced classes it can be misleading (the "nothing is pie" classifier on the slides). Our classes are balanced, so here accuracy and F1 are similar.

In [ ]:
from sklearn.metrics import accuracy_score            # share of correct predictions
from sklearn.metrics import classification_report     # precision, recall and F1 per class

acc = accuracy_score(test["party"], predictions)
print(f"Accuracy: {acc:.2f}")
print() # empty line for space
print(classification_report(test["party"], predictions))

The **confusion matrix**: rows are the true party, columns the predicted party. With R as positive: bottom right = true positives (TP), top right = false positives (FP), bottom left = false negatives (FN), top left = true negatives (TN).

In [ ]:
import matplotlib.pyplot as plt                       # plots
from sklearn.metrics import ConfusionMatrixDisplay    # draws a confusion matrix

ConfusionMatrixDisplay.from_predictions(test["party"], predictions)
plt.show()

💬 What do precision and recall for R mean here, in your own words?

✏️ In the `model = ...` cell of part 5, remove `stop_words="english", ` or change `min_df=2` to `min_df=10`. Run the cells again down to here. Does the accuracy change?

✏️ In part 3, change `test_size=0.2` to `test_size=0.5` and use *Runtime → Run after*. What changes?

## 7. Bag of words or TF-IDF?

Same split, same logistic regression, only step 1 is different: word counts (`CountVectorizer`) instead of TF-IDF. We report accuracy and the **macro F1** (the average of the F1 for D and for R).

In [ ]:
from sklearn.metrics import f1_score   # F1 score

# the same pipeline, but step 1 counts words (bag of words)

bow_model = make_pipeline(CountVectorizer(stop_words="english", min_df=2), LogisticRegression(max_iter=1000))
bow_model.fit(train["text"], train["party"])
bow_predictions = bow_model.predict(test["text"])

bow_acc = accuracy_score(test["party"], bow_predictions)
bow_f1 = f1_score(test["party"], bow_predictions, average="macro")     # macro: average of D and R
tfidf_f1 = f1_score(test["party"], predictions, average="macro")

print(f"Bag of words: accuracy {bow_acc:.2f}, F1 {bow_f1:.2f}")
print(f"TF-IDF:       accuracy {acc:.2f}, F1 {tfidf_f1:.2f}")
print() # empty line for space
print((bow_predictions != predictions).sum(), "of", len(test), "test texts get a different label")

## 8. Which words does the classifier use?

A positive weight points to R, a negative weight points to D.

1. `model[0]` is step 1 (TF-IDF): it gives us the words.
2. `model[1]` is step 2 (logistic regression): `.coef_[0]` gives us the weights.
3. We put words and weights together and sort them.

In [ ]:
words = model[0].get_feature_names_out()         # 1. the vocabulary of step 1
coefficients = model[1].coef_[0]                 # 2. one weight per word
weights = pd.Series(coefficients, index=words)   # 3. words and weights together

print(weights.sort_values().head(15))   # 3. the 15 most negative weights: typical for D

In [ ]:
print(weights.sort_values().tail(15))   # the 15 most positive weights: typical for R

✏️ Change `15` to `30` to see more words.

💬 Do these words show polarized language, or something else, for example names, states or standard phrases?

Note: **a classifier can be accurate without understanding anything about the politics**, e.g. because names and states give the party away.

## 9. Where does the classifier go wrong?

1. We add the predictions and the probability for R to the test table.
2. We keep the rows where the prediction is wrong.

A probability close to 0.5 means the model was unsure.

In [ ]:
test["pred"] = predictions                       # 1. predicted party
test["prob_R"] = probabilities[:, 1].round(2)    # 1. probability for R

errors = test[test["party"] != test["pred"]]     # 2. only the wrong predictions
print(len(errors), "wrong predictions")
pd.set_option("display.max_colwidth", None)    # show the full titles, not cut off
errors[["party", "pred", "prob_R", "title"]].head(10)

Two wrong predictions in full:

In [ ]:
for row in [0, 1]:                               # ✏️ change the numbers to read other errors
    print("True:", errors["party"].iloc[row], "| predicted:", errors["pred"].iloc[row], "| prob_R:", errors["prob_R"].iloc[row])
    print(errors["title"].iloc[row])
    print() # empty line for space
    print(errors["text"].iloc[row])              # the full press release
    print("-" * 80)                              # a line between the two examples

💬 What was the problem? For example: a bipartisan bill, a local grant announcement, words typical for the other party, a short or generic text, or quotes from the other side.